<a href="https://colab.research.google.com/github/contreras-juan/Material-Ciencia-de-Datos/blob/main/Deep_Learning/Ejercicios/03_Taller_RNN_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


<h1 style="color: #FECB05; text-align: center;">Taller práctico: RNN y LSTM</h1>


<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)


---


<h2 style="color: #007ACC;">Instrucciones</h2>

Este taller practica RNN/LSTM/GRU/ConvLSTM con **datos y ejercicios distintos** a los del cuaderno `07_RNN.ipynb`.
Aquí no reutilizamos manchas solares, pasajeros de aerolínea, la onda seno, ni las simulaciones de GRU/ConvLSTM de las notas.

Enfocamos:

1. Ventanas **multivariadas** y forma `(samples, timesteps, features)`
2. Detección de **fuga de información** en preprocesado
3. Serie caótica sintética (Mackey–Glass) con `SimpleRNN`
4. Temperaturas diarias con `LSTM` y elección de `look_back`
5. Pronóstico multipaso: enfoque **directo** vs **recursivo**
6. LSTM apilada + regularización
7. Multipaso con **baselines + LSTM single-shot** (estilo tutorial TensorFlow)
8. **GRU** en demanda eléctrica real (Alemania)
9. **ConvLSTM** en calidad del aire real (estaciones de Pekín)
10. Flujo completo en producción mensual de leche + baseline estacional

**Cómo trabajar:**

- Completa las celdas con `# Escribe tu código aquí`.
- Split siempre **temporal** (nunca `train_test_split` aleatorio sobre series).
- Escala con estadísticas **solo de train**.
- Reporta RMSE/MAE y, cuando aplique, curvas o gráficos de predicción.

**Tiempo sugerido:** 3.5–4.5 horas.


<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [Ejercicio 1. Ventanas multivariadas](#ejercicio-1)
- [Ejercicio 2. Cacería de leakage](#ejercicio-2)
- [Ejercicio 3. SimpleRNN en Mackey–Glass](#ejercicio-3)
- [Ejercicio 4. LSTM en temperaturas diarias](#ejercicio-4)
- [Ejercicio 5. Multipaso directo vs recursivo](#ejercicio-5)
- [Ejercicio 6. LSTM apilada y Dropout](#ejercicio-6)
- [Ejercicio 7. Multipaso con baselines y performance](#ejercicio-7)
- [Ejercicio 8. GRU en demanda eléctrica](#ejercicio-8)
- [Ejercicio 9. ConvLSTM en calidad del aire](#ejercicio-9)
- [Ejercicio integrador. Producción de leche](#ejercicio-integrador)


---


<h2 style="color: #007ACC;">Instalación e importaciones</h2>

Ejecuta primero la celda de instalación y luego la de importaciones.


In [ ]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib scikit-learn tensorflow


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Dense, Dropout, ConvLSTM2D, Conv2D, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)


In [ ]:
def temporal_split_arrays(*arrays, train_ratio=0.8):
    """Parte arrays ya alineados respetando el orden temporal."""
    n = len(arrays[0])
    cut = int(n * train_ratio)
    return [a[:cut] for a in arrays] + [a[cut:] for a in arrays]


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(np.asarray(y_true).ravel(), np.asarray(y_pred).ravel())))


def plot_history(history, title="Curvas de entrenamiento"):
    hist = history.history
    epochs = range(1, len(hist["loss"]) + 1)
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, hist["loss"], label="train loss")
    if "val_loss" in hist:
        plt.plot(epochs, hist["val_loss"], label="val loss")
    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


---

<a id="ejercicio-1"></a>
<h2 style="color: #007ACC;">Ejercicio 1. Ventanas multivariadas</h2>

**Objetivo:** construir ventanas con varias features por timestep.


En el material de clase la serie era univariada (`features=1`). Aquí trabajamos con una matriz de features.

1. Implementa `make_windows_multivariate(features, target, look_back)`:
   - `features` tiene shape `(T, F)`
   - `target` tiene shape `(T,)` (misma longitud)
   - cada muestra de $X$ usa `features[t:t+L]`
   - el target es `target[t+L]`
   - `X.shape = (N, look_back, F)` y `y.shape = (N,)`
2. Prueba con:
   - `features = [[i, i**2] for i in range(15)]`
   - `target = range(15)`
   - `look_back=4`
3. Imprime `X[:2]` y `y[:2]` y verifica a mano.

**Pregunta:** si quieres predecir la temperatura de mañana usando temperatura de hoy **y** el día del año, ¿qué va en `features` y qué en `target`?


In [ ]:
# Escribe tu código aquí
def make_windows_multivariate(features, target, look_back):
    pass


# Prueba y respuesta:


---

<a id="ejercicio-2"></a>
<h2 style="color: #007ACC;">Ejercicio 2. Cacería de leakage</h2>

**Objetivo:** detectar y corregir errores típicos de preprocesado en series.


Dataset (nacimientos diarios):

`https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-total-female-births.csv`

El siguiente pipeline tiene **al menos 3 errores**. No lo copies: **reescríbelo bien**.

```python
# PIPELINE INCORRECTO (solo para análisis)
df = pd.read_csv(URL)
series = df["Births"].values.astype("float32")
scaler = MinMaxScaler()
series = scaler.fit_transform(series.reshape(-1, 1)).ravel()   # (1)
X, y = [], []
L = 7
for i in range(len(series) - L):
    X.append(series[i:i+L])
    y.append(series[i+L])
X = np.asarray(X).reshape(-1, 1, L)                           # (2)
y = np.asarray(y)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(          # (3)
    X, y, test_size=0.2, random_state=42
)
```

1. Enumera los errores (1), (2) y (3) y explica el daño de cada uno.
2. Implementa la versión correcta: split temporal → scaler solo en train → ventanas con shape `(N, L, 1)`.
3. Reporta shapes finales de train/test.


In [ ]:
# Escribe tu código aquí
BIRTHS_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-total-female-births.csv"

# Errores:
# (1)
# (2)
# (3)


---

<a id="ejercicio-3"></a>
<h2 style="color: #007ACC;">Ejercicio 3. SimpleRNN en Mackey–Glass</h2>

**Objetivo:** entrenar una RNN clásica en una serie caótica (distinta a la onda seno del curso).


La serie de **Mackey–Glass** es un sistema dinámico con dependencia retardada; se usa mucho como banco de prueba de predicción no lineal.

1. Implementa (o completa) un generador discreto de Mackey–Glass y genera ~1200 puntos.
2. Escala solo con train (split temporal 80/20 **antes** de ventanas, o ventana y luego split temporal; sé consistente).
3. `look_back=30`, forma `(N, 30, 1)`.
4. Entrena `SimpleRNN(48) + Dense(1)` con `EarlyStopping`.
5. Grafica loss y real vs predicción en test; reporta RMSE (escala original si escalaste).

Fórmula discreta sugerida (paso unitario):

$$
x_{t+1} = x_t + \frac{0.2\, x_{t-\tau}}{1 + x_{t-\tau}^{10}} - 0.1\, x_t
$$

con $\tau=17$ y condición inicial constante (p. ej. $x_0=1.2$). Descarta un burn-in inicial (~200 pasos).


In [ ]:
# Escribe tu código aquí
def mackey_glass(n_points=1200, tau=17, burn_in=200, x0=1.2):
    pass


---

<a id="ejercicio-4"></a>
<h2 style="color: #007ACC;">Ejercicio 4. LSTM en temperaturas diarias</h2>

**Objetivo:** aplicar LSTM one-step a datos reales nuevos y razonar el `look_back`.


Dataset:

`https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv`

1. Carga la columna `Temp`.
2. Split temporal 80/20 + `StandardScaler` (o `MinMaxScaler`) ajustado solo en train.
3. Entrena **dos** LSTM one-step con el mismo presupuesto de épocas/callbacks:
   - Modelo A: `look_back=7`
   - Modelo B: `look_back=30`
4. Compara RMSE de test en escala original.
5. Grafica predicciones del mejor modelo en el tramo de test.

**Pregunta:** ¿qué `look_back` esperarías a priori para una serie con ciclo semanal vs estacionalidad anual? ¿Coincide con tu resultado?


In [ ]:
# Escribe tu código aquí
TEMPS_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"


---

<a id="ejercicio-5"></a>
<h2 style="color: #007ACC;">Ejercicio 5. Multipaso directo vs recursivo</h2>

**Objetivo:** contrastar dos estrategias de horizonte $H>1$ (distinto al único enfoque multi-output del curso).


Usa la serie de temperaturas (o una submuestra de ~1500 puntos para acelerar) con `look_back=14` y horizonte `H=7`.

1. **Directo (multi-output):** ventanas que mapean $L$ pasos → $H$ futuros; modelo `LSTM + Dense(H)`.
2. **Recursivo:** entrena un modelo one-step; para cada muestra de test, predice 1 paso, reinyecta la predicción en la ventana y repite $H$ veces.
3. Calcula RMSE **por horizonte** (días 1…7) en ambos enfoques.
4. Grafica ambas curvas de RMSE vs horizonte.

**Pregunta:** ¿en qué horizonte empieza a degradarse más el método recursivo y por qué?


In [ ]:
# Escribe tu código aquí


---

<a id="ejercicio-6"></a>
<h2 style="color: #007ACC;">Ejercicio 6. LSTM apilada y Dropout</h2>

**Objetivo:** experimentar con profundidad y regularización (no es la tabla RNN/LSTM/GRU del curso).


Sobre Mackey–Glass o temperaturas (elige uno y fíjalo):

1. Entrena un modelo **shallow**: `LSTM(32) + Dense(1)`.
2. Entrena un modelo **stacked**:
   - `LSTM(32, return_sequences=True)`
   - `Dropout(0.2)`
   - `LSTM(16)`
   - `Dense(1)`
3. Misma data, mismo `look_back`, mismos callbacks.
4. Tabla con `test_rmse` y `n_params`.
5. Comenta si la profundidad ayudó o solo sobreajustó.


In [ ]:
# Escribe tu código aquí


---

<a id="ejercicio-7"></a>
<h2 style="color: #007ACC;">Ejercicio 7. Multipaso con baselines y performance</h2>

**Objetivo:** reproducir la idea del [tutorial de series de TensorFlow](https://www.tensorflow.org/tutorials/structured_data/time_series#performance_3):
pronosticar **varios pasos futuros a la vez**, medir baselines fuertes y comparar performance.

En ese tutorial se predice el clima de Jena (24 h de historia → 24 h futuras) y se observa que, a menudo, modelos más complejos apenas mejoran a Dense/Conv.


Usaremos el dataset **Jena Climate** (mismo del tutorial), pero en versión reducida para el taller:

- Descarga el CSV del zip oficial de TensorFlow.
- Quédate con un subconjunto temporal (p. ej. las primeras ~25 000 filas horarias tras resampleo, o un muestreo cada hora si ya viene en 10 min).
- Usa solo algunas variables, p. ej. `T (degC)`, `p (mbar)`, `rh (%)` (o las que estén disponibles con nombres cercanos).

### Tareas

1. Carga y limpia Jena; resamplea a **1 hora** si hace falta.
2. Split temporal ~70/20/10 (train/val/test) **antes** de normalizar.
3. Normaliza restando media y dividiendo por std **calculadas solo en train** (como en el tutorial).
4. Construye ventanas multipaso:
   - `INPUT_WIDTH = 24`, `OUT_STEPS = 24`
   - `X.shape = (N, 24, F)`, `Y.shape = (N, 24, F)`
5. Implementa y evalúa (MAE) estos modelos **no entrenables**:
   - **Last:** repite el último timestep de la entrada durante `OUT_STEPS`.
   - **Repeat:** devuelve la ventana de entrada completa como predicción del día siguiente.
6. Entrena un **LSTM single-shot**:
   - `LSTM(...)` con `return_sequences=False`
   - `Dense(OUT_STEPS * F)`
   - `Reshape((OUT_STEPS, F))`
7. (Opcional) Añade un Dense single-shot (Lambda al último paso + Dense + Reshape).
8. Construye un gráfico de barras de MAE val/test por modelo (como la sección *Performance* del tutorial).
9. **Pregunta:** ¿tiene sentido la complejidad extra del LSTM frente a los baselines en *este* problema? Justifica con el gráfico.

> Referencia: [Time series forecasting — Performance (multi-step)](https://www.tensorflow.org/tutorials/structured_data/time_series#performance_3)


In [ ]:
# Escribe tu código aquí
JENA_ZIP_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"


---

<a id="ejercicio-8"></a>
<h2 style="color: #007ACC;">Ejercicio 8. GRU en demanda eléctrica</h2>

**Objetivo:** entrenar una GRU one-step sobre **datos reales** de electricidad (distintos a las series sintéticas de las notas) y compararla con LSTM.


Dataset (Open Power System Data — Alemania, diario 2006–2017):

`https://raw.githubusercontent.com/jenfly/opsd/master/opsd_germany_daily.csv`

Columnas: `Date`, `Consumption`, `Wind`, `Solar`, `Wind+Solar` (GWh).
`Consumption` está completa; viento/solar tienen huecos al inicio.

1. Carga el CSV y usa **`Consumption`** como serie objetivo.
2. Split temporal 80/20 + escalado **solo con train**.
3. Ventanas `look_back=14` (dos semanas) con forma `(N, 14, 1)`.
4. Entrena, con el mismo presupuesto (épocas, batch, callbacks, seed):
   - `GRU(32) + Dense(1)`
   - `LSTM(32) + Dense(1)`
5. Tabla con `test_rmse` en **escala original** y `n_params`.
6. Grafica real vs predicción GRU en el tramo de test.
7. **Pregunta:** ¿observas estacionalidad semanal? ¿GRU justifica (o no) su menor número de parámetros frente a LSTM?

**Extra (opcional):** a partir de 2012 (cuando hay `Wind` y `Solar` con menos NA), arma ventanas **multivariadas** `[Consumption, Wind, Solar]` para predecir `Consumption` al día siguiente.


In [ ]:
# Escribe tu código aquí
OPSD_URL = "https://raw.githubusercontent.com/jenfly/opsd/master/opsd_germany_daily.csv"


---

<a id="ejercicio-9"></a>
<h2 style="color: #007ACC;">Ejercicio 9. ConvLSTM en calidad del aire</h2>

**Objetivo:** aplicar ConvLSTM a un **panel espacial real**: PM2.5 horario de estaciones de Pekín, agregado a diario.


Dataset UCI *Beijing Multi-Site Air-Quality* (12 estaciones, 2013–2017):

`https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip`

El zip contiene otro zip con un CSV por estación (`PM2.5`, meteorología, etc.).

Usa **9 estaciones** en una grilla $3 \times 3$ según geografía aproximada (norte → sur, oeste → este):

| | Oeste | Centro | Este |
|---|---|---|---|
| Norte | Dingling | Changping | Huairou |
| Medio | Wanliu | Aotizhongxin | Shunyi |
| Sur | Gucheng | Tiantan | Dongsi |

### Tareas

1. Descarga y extrae el zip (ojo: hay un zip anidado).
2. Para cada estación de la grilla, construye la serie **diaria** de `PM2.5` (media del día; interpola NA cortos).
3. Apila las 9 series en un tensor `(T, 3, 3)`.
4. Split temporal ~75% train; normaliza con media/std **solo de train**.
5. Ventanas `look_back=7`:
   - `X.shape = (N, 7, 3, 3, 1)`
   - `y.shape = (N, 3, 3, 1)`  (mapa del día siguiente)
6. Compara:
   - **LSTM plana:** reshape a `(N, 7, 9)` → `LSTM` → `Dense(9)`.
   - **ConvLSTM2D:** mantiene la grilla → `ConvLSTM2D` → `Conv2D` (1 canal).
7. RMSE de test del mapa completo en µg/m³ (escala original) y `n_params`.
8. Visualiza un día de test: real / LSTM plana / ConvLSTM.
9. **Pregunta:** ¿tiene sentido el sesgo espacial aquí (contaminación que se comparte entre estaciones vecinas)? ¿ConvLSTM mejora, empata o pierde? Interpreta.

> No uses la grilla sintética de demanda de `07_RNN.ipynb`: este ejercicio es con mediciones reales.


In [ ]:
# Escribe tu código aquí
BEIJING_ZIP_URL = "https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip"

GRID = [
    ["Dingling", "Changping", "Huairou"],
    ["Wanliu", "Aotizhongxin", "Shunyi"],
    ["Gucheng", "Tiantan", "Dongsi"],
]


---

<a id="ejercicio-integrador"></a>
<h2 style="color: #007ACC;">Ejercicio integrador. Producción de leche</h2>

**Objetivo:** cerrar el flujo completo en un dataset mensual con estacionalidad, distinto a Airline Passengers.


Dataset:

`https://raw.githubusercontent.com/plotly/datasets/master/monthly-milk-production-pounds.csv`

Columnas: `Month`, `Monthly milk production (pounds per cow)`.

1. Split temporal (~75% train).
2. Escalado solo con train.
3. Ventanas `look_back=12` con forma `(N, 12, 1)`.
4. Entrena una LSTM one-step con EarlyStopping.
5. Reporta RMSE train/test en escala original.
6. Construye un **baseline estacional ingenuo**: predicción = valor de hace 12 meses (en el tramo de test).
7. Compara RMSE del baseline vs LSTM.
8. Conclusión breve: ¿la red supera a “el mismo mes del año pasado”? ¿dónde falla?

**Extra (opcional):** añade features cíclicas del mes (`sin/cos` del mes) y pasa a ventanas multivariadas del Ejercicio 1.


In [ ]:
# Escribe tu código aquí
MILK_URL = "https://raw.githubusercontent.com/plotly/datasets/master/monthly-milk-production-pounds.csv"


---

<h2 style="color: #007ACC;">Criterios de autoevaluación</h2>

| Criterio | Cumple |
|----------|--------|
| Implementaste ventanas multivariadas `(N, T, F)` | ☐ |
| Corregiste leakage (scaler/split/shape) con explicación | ☐ |
| Entrenaste SimpleRNN en Mackey–Glass con RMSE y gráfico | ☐ |
| Comparaste `look_back` 7 vs 30 en temperaturas | ☐ |
| Contrastaste multipaso directo vs recursivo | ☐ |
| Comparaste LSTM shallow vs stacked+Dropout | ☐ |
| Comparaste baselines Last/Repeat vs LSTM single-shot (Ej. 7) | ☐ |
| Comparaste GRU vs LSTM en demanda eléctrica alemana (Ej. 8) | ☐ |
| Comparaste LSTM plana vs ConvLSTM en PM2.5 de Pekín (Ej. 9) | ☐ |
| Completaste leche + baseline estacional con conclusión | ☐ |
